In [1]:
import os
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm

In [2]:
# load new classification
df = pd.read_table("/lustre/bio_running/GTEx_Long_Project/metadata/GTEx.chrM_info_extended.7S.reverse.reads.classified.v2.tsv.new_classification.tsv")
df

,Chr,ChrStart,ChrEnd,read_id,MAPQ,strand,read_length,alignment_length,reference_length,polyA_length,sample,overlapping_gene_ids,overlapping_gene_names,n_overlapping_gene_ids,chrM_primary_class,new_class
0,chrM,189,397,8219ade2-143f-4d8b-b15a-fc02f6430ea9,60,-,216.0,205,208,32,GTA1,7S,7S,1,complete_gene,fully_cleaved
1,chrM,189,396,4ed13e94-9fc0-4635-831b-d1ac2708c6ba,60,-,239.0,211,207,29,GTA1,7S,7S,1,complete_gene,fully_cleaved
2,chrM,189,397,0ae625d5-6c3e-4f83-ac35-9dc6731d3b9b,60,-,210.0,207,208,24,GTA1,7S,7S,1,complete_gene,fully_cleaved
3,chrM,189,397,cab401c4-514b-4b1f-8691-c46bb744c605,60,-,213.0,208,208,29,GTA1,7S,7S,1,complete_gene,fully_cleaved
4,chrM,189,398,66097d6c-e560-4a0e-913c-4fad5cd0c7bc,60,-,242.0,210,209,59,GTA1,7S,7S,1,complete_gene,fully_cleaved
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
789846,chrM,16388,16569,4f8cec63-475c-40f8-a615-a3e1796952d8,60,-,359.0,184,181,11,GTH9,NaN,NaN,0,non_coding_region,partially_cleaved_short
789847,chrM,16389,16569,5fe5947e-4bee-4f72-aa38-56b6b91b23ea,60,-,254.0,180,180,15,GTH9,NaN,NaN,0,non_coding_region,partially_cleaved_short
789848,chrM,16393,16569,1300a2cc-7fea-47ef-ab23-b5771b7e30fb,60,-,239.0,174,176,0,GTH9,NaN,NaN,0,non_coding_region,partially_cleaved_short
789849,chrM,16395,16564,46ce9b31-c444-4caa-8573-aece049e5431,60,-,182.0,169,169,4,GTH9,NaN,NaN,0,non_coding_region,partially_cleaved_short


In [3]:
df.new_class.value_counts()

partially_cleaved_short    369133
partially_cleaved_long     367166
fully_cleaved               53552
Name: new_class, dtype: int64

In [4]:
df[["sample", "new_class"]].value_counts()

sample  new_class              
GTA9    partially_cleaved_long     62147
        partially_cleaved_short    33341
GTC9    partially_cleaved_long     26662
GTG5    partially_cleaved_short    20857
GTH5    partially_cleaved_short    19300
                                   ...  
GTB1    partially_cleaved_long        39
GTF9    fully_cleaved                 37
GTD1    fully_cleaved                 27
GTF7    fully_cleaved                 25
GTB1    fully_cleaved                  4
Length: 219, dtype: int64

In [5]:
basepath = "/lustre/bio_running/GTEx_Long_Project/analysis"
samples = df["sample"].value_counts().index.to_list()

failed = []
for _,sample in enumerate(samples):
    print()
    print("\n######################################################################################################")
    print("######################################################################################################")
    print("######################################################################################################")
    print(sample, f"Total Processed samples: {_}/{len(samples)}")
    df_sample = df[df["sample"] == sample].copy()
    unique_reads = df_sample.read_id.value_counts().index.tolist()
    number_of_unique_reads = len(unique_reads)
    print("Shape of df-reads for current sample:", df_sample.shape)
    print("Unique number of reads:", number_of_unique_reads)
    display(df_sample.head(2))
    print("CHECKING number of reads:")
    if not df_sample.shape[0] == number_of_unique_reads:
        print("!!!!!!!!ERROR!!!!!!!!!!! ")
        break
    mitoBAM_fp = os.path.join(basepath, sample, f"{sample}.genome.chrM.reverse.bam")
    print("Starting mito-BAM filepath:", mitoBAM_fp)
    for new_class in ["fully_cleaved", "partially_cleaved_short", "partially_cleaved_long"]:
        print(f"\n#####\nElaborating class: {new_class}")
        df_sample_current_class = df_sample.query(f"new_class == '{new_class}'")
        if df_sample_current_class.empty:
            print("!!! Problem with current class !!! Empty reads-list for this sample/class combination.")
            failed.append(f"{sample}|{new_class}")
            continue
        display(df_sample_current_class[["sample", "new_class"]].value_counts())
        reads_list_current_class = df_sample_current_class.read_id
        print(df_sample_current_class.shape)
        print(f"Number of reads for current class: {len(reads_list_current_class)}")
        out_mitoBAM_fp = mitoBAM_fp + f".{new_class}.bam"
        out_mitoBAM_readslist_fp = out_mitoBAM_fp + ".readslist"
        out_mitoBAM_depth_fp = out_mitoBAM_fp + ".depth.txt"
        print(f"- output mitoBAM for current class: {out_mitoBAM_fp}")
        print(f"- output mitoBAM readslist for current class: {out_mitoBAM_readslist_fp}")
        print(f"- output mitoBAM depth file for current class: {out_mitoBAM_depth_fp}")
        # write readslist on disk
        with open(out_mitoBAM_readslist_fp, "w") as out_mitoBAM_readslist:
            for read in reads_list_current_class:
                out_mitoBAM_readslist.write(read + "\n")
        os.system(f"cat {out_mitoBAM_readslist_fp} | wc -l")
        # create sub-mitoBAM file, indexing it, creating corresponding depth file
        os.system(f"samtools view -F 2308 -@ 24 -b -N {out_mitoBAM_readslist_fp} {mitoBAM_fp} > {out_mitoBAM_fp}")
        os.system(f"samtools index -@ 24 {out_mitoBAM_fp}")
        os.system(f"samtools depth -a -@ 24 {out_mitoBAM_fp} > {out_mitoBAM_depth_fp}")
        os.system(f"samtools view -@ 24 {out_mitoBAM_fp} | cut -f 2,3 | sort | uniq -c | sort -nr")
        os.system(f"wc -l {out_mitoBAM_depth_fp}")

print("\n\n\nComputation Finised!")
print(f"Total Processed samples: {_+1}/{len(samples)}")
print(f"Failed sample|new_class combinations: {len(failed)}")



######################################################################################################
######################################################################################################
######################################################################################################
GTA9 Total Processed samples: 0/73
Shape of df-reads for current sample: (99127, 16)
Unique number of reads: 99127


,Chr,ChrStart,ChrEnd,read_id,MAPQ,strand,read_length,alignment_length,reference_length,polyA_length,sample,overlapping_gene_ids,overlapping_gene_names,n_overlapping_gene_ids,chrM_primary_class,new_class
4059,chrM,189,395,ab175762-42da-4f10-8966-6032a392965c,60,-,225.0,206,206,62,GTA9,7S,7S,1,complete_gene,fully_cleaved
4060,chrM,189,396,a265af16-4b83-4fdc-bdb9-cd3b65328196,60,-,233.0,208,207,10,GTA9,7S,7S,1,complete_gene,fully_cleaved


CHECKING number of reads:
Starting mito-BAM filepath: /lustre/bio_running/GTEx_Long_Project/analysis/GTA9/GTA9.genome.chrM.reverse.bam

#####
Elaborating class: fully_cleaved


sample  new_class    
GTA9    fully_cleaved    3639
dtype: int64

(3639, 16)
Number of reads for current class: 3639
- output mitoBAM for current class: /lustre/bio_running/GTEx_Long_Project/analysis/GTA9/GTA9.genome.chrM.reverse.bam.fully_cleaved.bam
- output mitoBAM readslist for current class: /lustre/bio_running/GTEx_Long_Project/analysis/GTA9/GTA9.genome.chrM.reverse.bam.fully_cleaved.bam.readslist
- output mitoBAM depth file for current class: /lustre/bio_running/GTEx_Long_Project/analysis/GTA9/GTA9.genome.chrM.reverse.bam.fully_cleaved.bam.depth.txt
3639
   3639 16	chrM
16569 /lustre/bio_running/GTEx_Long_Project/analysis/GTA9/GTA9.genome.chrM.reverse.bam.fully_cleaved.bam.depth.txt

#####
Elaborating class: partially_cleaved_short


...

(4, 16)
Number of reads for current class: 4
- output mitoBAM for current class: /lustre/bio_running/GTEx_Long_Project/analysis/GTB1/GTB1.genome.chrM.reverse.bam.fully_cleaved.bam
- output mitoBAM readslist for current class: /lustre/bio_running/GTEx_Long_Project/analysis/GTB1/GTB1.genome.chrM.reverse.bam.fully_cleaved.bam.readslist
- output mitoBAM depth file for current class: /lustre/bio_running/GTEx_Long_Project/analysis/GTB1/GTB1.genome.chrM.reverse.bam.fully_cleaved.bam.depth.txt
4
      4 16	chrM
16569 /lustre/bio_running/GTEx_Long_Project/analysis/GTB1/GTB1.genome.chrM.reverse.bam.fully_cleaved.bam.depth.txt

#####
Elaborating class: partially_cleaved_short


sample  new_class              
GTB1    partially_cleaved_short    40
dtype: int64

(40, 16)
Number of reads for current class: 40
- output mitoBAM for current class: /lustre/bio_running/GTEx_Long_Project/analysis/GTB1/GTB1.genome.chrM.reverse.bam.partially_cleaved_short.bam
- output mitoBAM readslist for current class: /lustre/bio_running/GTEx_Long_Project/analysis/GTB1/GTB1.genome.chrM.reverse.bam.partially_cleaved_short.bam.readslist
- output mitoBAM depth file for current class: /lustre/bio_running/GTEx_Long_Project/analysis/GTB1/GTB1.genome.chrM.reverse.bam.partially_cleaved_short.bam.depth.txt
40
     40 16	chrM
16569 /lustre/bio_running/GTEx_Long_Project/analysis/GTB1/GTB1.genome.chrM.reverse.bam.partially_cleaved_short.bam.depth.txt

#####
Elaborating class: partially_cleaved_long


sample  new_class             
GTB1    partially_cleaved_long    39
dtype: int64

(39, 16)
Number of reads for current class: 39
- output mitoBAM for current class: /lustre/bio_running/GTEx_Long_Project/analysis/GTB1/GTB1.genome.chrM.reverse.bam.partially_cleaved_long.bam
- output mitoBAM readslist for current class: /lustre/bio_running/GTEx_Long_Project/analysis/GTB1/GTB1.genome.chrM.reverse.bam.partially_cleaved_long.bam.readslist
- output mitoBAM depth file for current class: /lustre/bio_running/GTEx_Long_Project/analysis/GTB1/GTB1.genome.chrM.reverse.bam.partially_cleaved_long.bam.depth.txt
39
     39 16	chrM
16569 /lustre/bio_running/GTEx_Long_Project/analysis/GTB1/GTB1.genome.chrM.reverse.bam.partially_cleaved_long.bam.depth.txt



Computation Finised!
Total Processed samples: 73/73
Failed sample|new_class combinations: 0
